In [59]:
import re

class QAItem:
    def __init__(self, question: str, answer: str):
        self.question = question.strip()
        self.answer = answer.strip()

    def to_html(self) -> str:
        return f"""
        <div class="qa-block">
            <div class="question">
                <h3>Question</h3>
                <p>{self.question}</p>
            </div>
            <div class="answer">
                <h3>Answer</h3>
                <p>{self.answer}</p>
            </div>
        </div>
        """


def parse_raw_text(text: str):
    """
    Parse text containing repeated blocks:
    Q : ...
    A : ...
    Returns a list of QAItem objects.
    """
    pattern = r"Q\s*:\s*(.*?)\s*A\s*:\s*(.*?)(?=Q\s*:|$)"
    matches = re.findall(pattern, text, re.DOTALL)

    qa_items = []
    for q, a in matches:
        qa_items.append(QAItem(q, a))
    return qa_items


def generate_html_page(qa_items):
    """Generate a full HTML page from a list of QAItem objects."""
    blocks = "\n".join(item.to_html() for item in qa_items)

    html = f"""
    <html>
    <head>
    <style>
        .qa-block {{
            border: 1px solid #ccc;
            padding: 15px;
            margin-bottom: 20px;
            border-radius: 6px;
            font-family: Arial, sans-serif;
        }}
        .question {{
            background-color: #eef6ff;
            padding: 10px;
            border-left: 4px solid #007acc;
            margin-bottom: 10px;
        }}
        .answer {{
            background-color: #f9f9f9;
            padding: 10px;
            border-left: 4px solid #666;
        }}
        h3 {{
            margin-top: 0;
        }}
    </style>
    </head>

    <body>
    {blocks}
    </body>
    </html>
    """
    return html


# Example: using your exact raw text
raw_text = """
Q : Expliquer l'équation de dirac

A : # L'équation de Dirac
L'équation de Dirac...

Q : Je suppose l'équation suivante
ds^2 =(c.dt)^2 +(i.v.dt)^2
Comment s'articulerait-elle avec Dirac ?

A : # Une approche intéressante...
Votre proposition inverse la convention habituelle...
"""

qa_list = parse_raw_text(raw_text)
html_output = generate_html_page(qa_list)

print(html_output)



    <html>
    <head>
    <style>
        .qa-block {
            border: 1px solid #ccc;
            padding: 15px;
            margin-bottom: 20px;
            border-radius: 6px;
            font-family: Arial, sans-serif;
        }
        .question {
            background-color: #eef6ff;
            padding: 10px;
            border-left: 4px solid #007acc;
            margin-bottom: 10px;
        }
        .answer {
            background-color: #f9f9f9;
            padding: 10px;
            border-left: 4px solid #666;
        }
        h3 {
            margin-top: 0;
        }
    </style>
    </head>

    <body>
    
        <div class="qa-block">
            <div class="question">
                <h3>Question</h3>
                <p>Expliquer l'équation de dirac</p>
            </div>
            <div class="answer">
                <h3>Answer</h3>
                <p># L'équation de Dirac
L'équation de Dirac...</p>
            </div>
        </div>
        

        <div c

In [60]:
import re

class QAItem:
    def __init__(self, question: str, answer: str):
        self.question = question.strip()
        self.answer = answer.strip()

    def to_html(self) -> str:
        return f"""
        <div class="qa-block">
            <div class="question">
                <h3>Question</h3>
                {self.question}
            </div>
            <div class="answer">
                <h3>Answer</h3>
                {self.answer}
            </div>
        </div>
        """


class TextTransformer:

    @staticmethod
    def convert_markers_to_h3(text: str) -> str:
        """
        Convert lines beginning with ## to <h3>...</h3>.
        """
        lines = text.split("\n")
        new_lines = []
        for line in lines:
            if line.strip().startswith("## "):
                content = line.strip()[3:]
                new_lines.append(f"<h3>{content}</h3>")
            elif line.strip().startswith("### "):
                content = line.strip()[4:]
                new_lines.append(f"<h4>{content}</h4>")
            elif line.strip().startswith("#### "):
                content = line.strip()[5:]
                new_lines.append(f"<h4>{content}</h4>")
            else:
                new_lines.append(line)
        return "\n".join(new_lines)

    @staticmethod
    def convert_bold_italic(text: str) -> str:
        """
        Convert **text** to <b><i>text</i></b>.
        """
        return re.sub(r"\*\*(.*?)\*\*", r"<b><i>\1</i></b>", text)

    @staticmethod
    def convert_pipe_tables(text: str) -> str:
        """
        Convert sequences of lines like:

        | A | B | C |
        | 1 | 2 | 3 |
        | 4 | 5 | 6 |

        into a single HTML table.
        """

        lines = text.split("\n")
        result = []
        table_buffer = []

        def flush_table():
            if not table_buffer:
                return ""
            rows_html = []
            for row in table_buffer:
                # skip rows where all cells are just dashes
                cells = [c.strip() for c in row.strip("|").split("|")]
                if all(re.fullmatch(r"-+", c) for c in cells):
                    continue
                rows_html.append("<tr>" + "".join(f"<td>{c}</td>" for c in cells) + "</tr>")
            if not rows_html:
                return ""
            return "<table>\n" + "\n".join(rows_html) + "\n</table>"

        for line in lines:
            stripped = line.strip()
            if stripped.startswith("|") and stripped.endswith("|"):
                table_buffer.append(stripped)
            else:
                if table_buffer:
                    result.append(flush_table())
                    table_buffer = []
                result.append(line)

        if table_buffer:
            result.append(flush_table())

        return "\n".join(result)

    @staticmethod
    def convert_newlines_to_p(text: str) -> str:
        """
        Convert blank-line-separated blocks into paragraphs.
        """
        paragraphs = text.split("\n\n")
        html_paragraphs = []
        for para in paragraphs:
            if para.strip().startswith("<table"):
                html_paragraphs.append(para)
            else:
                html_paragraphs.append(f"<p>{para.strip()}</p>")
        return "\n".join(html_paragraphs)

    @staticmethod
    def convert_unordered_lists(text: str) -> str:
        """
        Convert consecutive lines starting with '- ' into <ul><li>...</li></ul>.
        """
        lines = text.split("\n")
        result = []
        ul_buffer = []

        def flush_ul():
            if not ul_buffer:
                return ""
            html = "<ul>\n" + "\n".join(f"<li>{item}</li>" for item in ul_buffer) + "\n</ul>"
            return html

        for line in lines:
            stripped = line.strip()
            if stripped.startswith("- "):
                ul_buffer.append(stripped[2:].strip())
            else:
                if ul_buffer:
                    result.append(flush_ul())
                    ul_buffer = []
                result.append(line)

        if ul_buffer:
            result.append(flush_ul())

        return "\n".join(result)

    @staticmethod
    def apply_all(text: str) -> str:
        text = TextTransformer.convert_markers_to_h3(text)
        text = TextTransformer.convert_bold_italic(text)
        text = TextTransformer.convert_pipe_tables(text)
        text = TextTransformer.convert_unordered_lists(text)
        text = TextTransformer.convert_newlines_to_p(text)
        return text


def load_text(filename: str) -> str:
    """
    Load text from a .txt file.
    """
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()

def save_html(filename: str, html_content: str):
    """
    Save HTML into a .html file.
    """
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"Saved HTML to: {filename}")

def parse_raw_text(text: str):
    """
    Parse text containing:
    Q : question
    A : answer
    """
    pattern = r"Q\s*:\s*(.*?)\s*A\s*:\s*(.*?)(?=Q\s*:|$)"
    matches = re.findall(pattern, text, re.DOTALL)
    return [QAItem(
        TextTransformer.apply_all(q),
        TextTransformer.apply_all(a)
    ) for q, a in matches]


def generate_html_page(qa_items):
    """
    Build a complete HTML page containing all Q/A blocks.
    """
    blocks = "\n".join(item.to_html() for item in qa_items)

    html = f"""
    <html>
    <head>
    <meta charset="UTF-8">
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 25px;
        }}
        .qa-block {{
            border: 1px solid #ccc;
            padding: 15px;
            margin-bottom: 20px;
            border-radius: 6px;
        }}
        .question {{
            background-color: #eef6ff;
            padding: 10px;
            border-left: 4px solid #007acc;
            margin-bottom: 10px;
        }}
        .answer {{
            background-color: #f9f9f9;
            padding: 10px;
            border-left: 4px solid #666;
        }}
        table {{
            border-collapse: collapse;
            margin: 10px 0;
        }}
        td {{
            border: 1px solid #444;
            padding: 5px 10px;
        }}
    </style>
    </head>

    <body>
    {blocks}
    </body>
    </html>
    """
    return html


# Example workflow:
#
# raw = load_text("input.txt")
# qa = parse_raw_text(raw)
# html = generate_html_page(qa)
# save_html("output.html", html)


In [61]:
# Example usage:
# ----------------------
file_name = "Conscience, qui es tu"
#file_name = "Why do we sleep"
file_name = "Conscience participative et réalité"

raw_text = load_text(fr"C:\Users\jeand\Desktop\Claude AI\{file_name}.txt")
qa_items = parse_raw_text(raw_text)
html_output = generate_html_page(qa_items)
save_html(fr"C:\Users\jeand\Desktop\Claude AI\{file_name}.html", html_output)

Saved HTML to: C:\Users\jeand\Desktop\Claude AI\Conscience participative et réalité.html
